# Fault report

Did staging clean the injected faults correctly? Compares the clean warehouse with the damaged one, using the fault log as ground truth.

1. Where did the rows go?
2. How accurate were the fills?
3. Does the analysis change?

In [5]:
import duckdb
import pandas as pd

RUN_ID = '1cee3a4af0f2'

con = duckdb.connect('../warehouse/hexstrat.duckdb', read_only=True)
con.execute("attach '../warehouse/hexstrat_faulty.duckdb' as faulty (read_only)")

fault_log = pd.read_parquet(f'../data/raw_faulty/fault_log_{RUN_ID}.parquet')
con.register('fault_log', fault_log)

BinderException: Binder Error: Failed to attach database: database with name "faulty" already exists

## 1. Where did the rows go?

In [6]:
con.sql('select count(*) from raw.moves').df()

,count_star()
0,8447807


In [7]:
con.sql('select count(*) from staging.stg_moves').df()

,count_star()
0,8447807


In [8]:
con.sql('select count(*) from faulty.raw.moves').df()

,count_star()
0,8490046


In [9]:
con.sql('select count(*) from faulty.staging.stg_moves').df()

,count_star()
0,8439406


In [10]:
len(fault_log[fault_log['fault_type'] == 'duplicate'])

42239

In [11]:
len(fault_log[fault_log['fault_type'] == 'missing'])

42450

In [12]:
len(fault_log)

84689

In [13]:
len(fault_log[(fault_log['column'] == 'game_id') | (fault_log['column'] == 'move_index')])

8486

In [14]:
len(fault_log[(fault_log['fault_type'] == 'duplicate') & (fault_log['column'] != 'game_id') & (fault_log['column'] != 'move_index')])

42239

In [15]:
len(fault_log[(fault_log['column'] == 'game_id') | (fault_log['column'] == 'move_index')]) + len(fault_log[(fault_log['fault_type'] == 'duplicate') & (fault_log['column'] != 'game_id') & (fault_log['column'] != 'move_index')])

50725

In [47]:
joined = con.sql("""select f2.column, original_value, 
case f2.column
when 'player' then f1.player
when 'points' then cast(f1.points as varchar)
when 'res_brick' then cast(f1.res_brick as varchar)
when 'res_wood' then cast(f1.res_wood as varchar)
when 'res_sheep' then cast(f1.res_sheep as varchar)
when 'res_wheat' then cast(f1.res_wheat as varchar)
when 'res_ore' then cast(f1.res_ore as varchar)
end as filled_vals

from fault_log f2
join faulty.staging.stg_moves f1 using (game_id, move_index) 
where f2.fault_type = 'missing' and
f2.column not in ('game_id', 'move_index', 'action_type')
""").df()
sum(joined['original_value'] == joined['filled_vals']) / len(joined)

0.8592188919164396

In [56]:
joined['correct'] = joined['original_value'] == joined['filled_vals']
joined.groupby('column')['correct'].mean().sort_values(ascending = False)

column
player       1.000000
points       0.977581
res_ore      0.833096
res_brick    0.832458
res_sheep    0.796825
res_wheat    0.794417
res_wood     0.784690
Name: correct, dtype: float64

In [59]:
notFaulty = con.sql('select * from marts.fct_opening_placements').df()
notFaulty.groupby('player')['won'].mean()

player
BLUE      0.442944
ORANGE    0.187519
RED       0.188019
WHITE     0.181518
Name: won, dtype: float64

In [60]:
faulty = con.sql('select * from faulty.marts.fct_opening_placements').df()
faulty.groupby('player')['won'].mean()

player
BLUE      0.442864
ORANGE    0.187484
RED       0.187932
WHITE     0.181518
Name: won, dtype: float64